# 1.实体记忆组件示例

知识点讲解：实体记忆（结构化输出 + 显式实体存储）

1. 什么是实体记忆（Entity Memory）
   - 实体记忆跟踪对话中提到的实体，并记住关于特定实体的既定事实：它用 LLM 从对话中提取实体信息，再随时间建立对该实体的知识
   - 一般用来存储和查询对话中引用的各类信息，例如人物、地点、事件等
   - 它不保存「完整对话流」，而是从对话中抽取出「实体 → 描述」的知识条目，例如从
     「我是慕小课，正在学习 LangChain」抽取出：
       {"慕小课": "用户本人，正在学习 LangChain", "LangChain": "用户正在学习的框架"}
   - 本质：把非结构化的对话，转化为结构化的「用户画像 / 知识库」

2. 实体记忆与其他记忆策略的本质区别
   策略                 存储形式            token 特征        信息组织
   缓冲记忆             消息列表原文        随轮次线性增长    时间序
   窗口记忆（第10章）   最近 N 条原文        恒定              时间序（丢弃旧的）
   摘要记忆（第11章）   摘要文本 + 近期原文  恒定              时间序（压缩旧的）
   实体记忆（本章）     实体字典            随实体数增长      按实体聚合
   - 前三者都是「按时间」组织，实体记忆是「按主题/对象」组织
   - 实体记忆能长期稳定记住「用户是谁、喜欢什么」，不会因对话变长而遗忘
   - 缺点：丢失对话的时序与上下文细节，无法回答「刚才我说了什么」

3. LangChain 实现范式的演进（1.x 显式 vs 0.x 隐式）
   - 0.x 封装了 ConversationEntityMemory，用内置固定提示词隐式抽取实体，
     但预设 Prompt 过于笨重、极度消耗 Token、对大模型要求极高，实用度并不高
   - 1.x 改为显式三步：
     a. 用 Pydantic 定义实体的数据结构（Entities）
     b. 用 llm.with_structured_output(Entities) 做类型安全的抽取
     c. 实体存储由应用层自己持有（本例是一个普通 dict）
   - 好处：结构可控、抽取可测、存储可换（dict → Redis → 图数据库）

4. with_structured_output 的工作原理（关键知识点）
   - 签名：llm.with_structured_output(schema) -> Runnable[..., schema 实例]
   - 底层机制：把 Pydantic 模型转换成 JSON Schema，
               通过模型的 function calling / tool calling 能力约束输出格式
   - 返回：一个新的 Runnable，其 invoke 直接返回 Pydantic 对象实例，
           而非 AIMessage，因此后面不需要接 OutputParser
   - 优势：输出结构由模型侧强制保证，比「提示词要求返回 JSON + 手动解析」可靠得多
   - 注意：要求模型支持 function calling，不支持的模型需退化为 JSON 模式

5. Pydantic 模型在结构化输出中的三个作用
   - 类型约束：dict[str, str] 明确要求「字符串到字符串的映射」
   - 语义提示：类的 docstring 与 Field(description=...) 会被写入 JSON Schema，
               直接影响模型的抽取行为，相当于「字段级提示词」
   - 运行时校验：模型返回的数据会被 Pydantic 校验，不符合类型会抛异常

6. temperature=0 的必要性
   - 实体抽取是「确定性任务」，同样的输入应得到同样的输出
   - temperature=0 关闭采样随机性，让结果尽可能稳定可复现
   - 对比：闲聊、创作类任务需要 temperature > 0 保证多样性
   - 本例中 temperature=0 同时作用于抽取与对话两条链（共用同一个 llm 实例）

7. 本示例的双链架构（重要设计模式）
   - 一轮对话需要两次 LLM 调用，分别由两条链承担：
     * entity_extractor：抽取链，负责「从输入中提取结构化实体」
         extract_prompt | llm.with_structured_output(Entities) → 输出 Entities 对象
     * chat_chain：对话链，负责「结合实体与历史生成回复」
         chat_prompt | llm | StrOutputParser() → 输出 str
   - 两条链共用同一个 llm 实例，但输出类型完全不同

8. dict.update 实现的实体累积与覆盖语义
   - entity_store.update(new_entities) 的行为：
       新实体名 → 新增条目
       已存在的实体名 → 用新描述「覆盖」旧描述
   - 覆盖是双刃剑：
       优点：实体信息可以随对话演进而更新（如用户换了城市）
       缺点：旧描述被直接丢弃，可能丢失历史信息
   - 若需累积而非覆盖，应改为手动合并描述文本

9. 典型抽取结果与多轮累积（来自课程示例）
   - 经过「我是慕小课…」「最喜欢的编程语言是 Python」「我住在广州」三轮对话后，
     entity_store 会累积出四条实体：
       {'慕小课': '慕小课最近正在学习LangChain。',
        'LangChain': 'LangChain 是一个专注于构建和连接语言模型的项目。',
        'Python': 'Python 是一门非常受欢迎且功能强大的编程语言…',
        '广州': '广州是中国的第三大城市…'}
   - 这正是实体记忆「按主题聚合、长期不遗忘」特征的直观体现

10. 实体记忆 + 消息历史的混合使用
   - 本示例同时维护了两份记忆：entity_store（实体）与 chat_history（对话流）
   - 这是生产环境的常见做法：
       entity_store 提供「长期稳定的用户画像」，永不遗忘
       chat_history 提供「近期对话的上下文」，可配合 trim_messages 裁剪
   - 两者互补：前者答「我是谁」，后者答「刚才聊了什么」

11. 注意事项
   - 每轮对话要调用两次 LLM（抽取 + 回复），延迟与成本约为普通对话的两倍
   - 抽取链可以异步化或降级到小模型，以降低延迟与成本
   - `entity_store or "无"` 利用了空 dict 的 falsy 特性，避免在提示词里渲染出 "{}"
   - entity_store 是普通 dict，进程退出即丢失，生产环境需持久化
   - 抽取质量高度依赖 system_prompt 与 Field description，需要针对业务调优


In [ ]:
import dotenv
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
dotenv.load_dotenv()
llm = ChatOpenAI(model="deepseek-v4-pro", temperature=0)

In [ ]:
class Entities(BaseModel):
    """从用户输入中抽取出的实体及其描述"""
    entities: dict[str, str] = Field(
        description="实体名称到实体描述的映射，没有实体时返回空字典",
    )

In [ ]:
extract_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是实体抽取助手，请从用户输入中抽取人物、地点、技术等实体，并给出简短描述。"),
    ("human", "{input}"),
])
entity_extractor = extract_prompt | llm.with_structured_output(Entities)
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是DeepSeek开发的聊天机器人，请结合已知实体信息回复用户。\n\n已知实体信息：\n{entities}"),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])
chat_chain = chat_prompt | llm | StrOutputParser()
entity_store: dict[str, str] = {}
chat_history = InMemoryChatMessageHistory()

In [ ]:
def chat(user_input: str) -> str:
    """一轮对话：先抽取实体写入存储，再带着实体信息与历史生成回复
    chat(user_input: str) -> str
      作用：完成一轮完整的「实体记忆对话」
      参数：
        user_input: str - 用户本轮的输入文本
      返回：
        str - AI 生成的回复文本
      副作用（三处状态变更）：
        1. entity_store 被更新（新增或覆盖实体条目）
        2. chat_history 追加一条 HumanMessage
        3. chat_history 追加一条 AIMessage
      LLM 调用次数：2 次
        第 1 次：抽取实体（entity_extractor）
        第 2 次：生成回复（chat_chain）
        因此延迟与成本约为普通单链对话的两倍
      执行顺序要点：
        必须「先抽取、再回复」，这样本轮刚提到的实体就能立即被用于本轮回复，
        实现「我是慕小课」→ 下一句就能称呼用户为慕小课的效果
    """
    entity_store.update(entity_extractor.invoke({"input": user_input}).entities)
    content = chat_chain.invoke({
        "input": user_input,
        "entities": entity_store or "无",
        "history": chat_history.messages,
    })
    chat_history.add_user_message(user_input)
    chat_history.add_ai_message(content)
    return content

In [ ]:
print(chat("你好，我是慕小课。我最近正在学习LangChain。"))
print(chat("我最喜欢的编程语言是 Python。"))
print(chat("我住在广州"))
print(entity_store)